    RICCARDO DEIDDA M: 70/90/00639

# Sperimentation Section
This section presents the preliminary experiments conducted prior to the design of the final face recognition system.

First of all, it was necessary to download the project dataset and analyze its contents.

In [ ]:
# PACKAGES INSTALLATION

from IPython.display import clear_output

!pip install pydrive
!pip install dlib
!pip install opencv-python
!pip install scikit-image

clear_output()

print("Packages have been installed!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import os
import dlib


!gdown --id "11V4qLWcpa2Njumt1g8t44sBqe4IBnk0K" -O faceRecognDataset.zip
!unzip faceRecognDataset.zip


clear_output()
print("Files have been downloaded!")

Now lets check the dataset contents.

In [ ]:

def plot_probe(probe_path):
  # this utiltiy function reads the input image through cv2 and plots it with
  # matplotlib pyplot.

  probe_img = cv2.imread(probe_path)
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)
  fig1, axes = plt.subplots(1,figsize=(3, 3))
  axes.imshow(probe_img)
  axes.set_title("Probe Image", fontweight="bold")
  axes.axis("off")
  plt.show()
  print("\n")

In [ ]:
# lets plot the probe image
probe_path = "../content/Masked_Recognition/probe.jpg"
plot_probe(probe_path)

# and the total number of reference inside the dataset
num_images = len(os.listdir("../content/Masked_Recognition/Reference/"))
print(f"\nThe total numer of images inside the reference dataset is: {num_images}")


By inspecting the images provided, I observed that the dataset consists of a single probe image of a masked individual together with 81 reference samples, each corresponding to a different identity.

##BSIF Implementation
The goal of this project is to develop a face recognition algorithm capable of correctly identifying individuals even when they are wearing protective masks, which introduce partial occlusion to the facial region. As a first step, it was necessary to implement a baseline recognition algorithm in order to evaluate different feature extraction methods and their robustness to occlusion.
In general, feature extraction for face recognition can be approached through three main categories:

- Handcrafted features
- Texture descriptors
- Convolutional Neural Networks

Traditional handcrafted approaches include methods such as the Haar Cascade Classifier or Histogram of Oriented Gradients. These techniques rely on explicitly designed low-level descriptors to capture edge or shape information. While computationally efficient, they are relatively limited in their ability to discriminate between identities under challenging conditions, so they re probably unsuitable for our requirements.

A better idea could be relying on texture descriptors. Techniques such as Eigenfaces, Local Binary Patterns and Binarized Statistical Image Features aim to capture textural patterns in the face region. These descriptors often achieve better performance than basic handcrafted methods while retaining relatively low computational complexity

Given the need for a method that is both computationally efficient and more robust than traditional handcrafted approaches such as Haar Cascades or HOG, texture descriptors represent a more suitable choice.
Because of this i hve chosen BSIF as a starting point for this project. The choiche of BSIF over Eigenfaces or Local Binary Pattern was made because BSIF can be considered an extension of LBP and it is documented to provide more discriminative textural informations.

### Image enhancement and normalization
Before implementing the BSIF-based recognition algorithm, the first essential step is to enhance and normalize both the dataset images and the probe. Proper normalization ensures that all face samples are comparable in terms of scale, orientation, and illumination, while enhancement can improve the visibility of  textural details that the feature extraction algorithm relies on.

The most reliable approach for preprocessing face images is to use RetinaFace. However since the dataset provided in this project already consists of aligned and uniformly sized images, retina face should not be strictly necessary but it can still be implemented for future application in other datasets.

In [ ]:
!pip install retina-face
clear_output()

print("Retina Face installed!")

In [ ]:
from retinaface import RetinaFace
import imutils
import matplotlib.pyplot as plt

def retina_normalization(img):
  # retina_normalization leverage retinaface to align and enhance the
  # input image

  retina_output = RetinaFace.extract_faces(img, align=True)

  if not retina_output:
    return None

  h,w,_ = img.shape
  img_normalized = retina_output[0]
  img_normalized = cv2.resize(img_normalized, (w,h))
  return img_normalized

In [ ]:
test_img = cv2.imread("../content/Masked_Recognition/Reference/100.jpg")
aligned = retina_normalization(test_img)


fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original")
ax[0].axis("off")
if aligned is not None:
  ax[1].imshow(aligned)
  ax[1].set_title("Aligned")
  ax[1].axis("off")

else:
  ax[1].set_title("No face detected")
  ax[1].axis("off")

plt.show()

Sadly, after several tests I observed that RetinaFace was not able to reliably detect faces in certain dataset images (for example, 383.jpg). As a result, adopting RetinaFace as the preprocessing stage would cause some images to be discarded, leading to an undesirable loss of data, since each identity is represented by only one sample.

Fortunately this issue does not represent a major limitation in the current project configuration because the dataset images are already aligned.

In [ ]:
test_img = cv2.imread("../content/Masked_Recognition/Reference/383.jpg")
aligned = retina_normalization(test_img)


fig, ax = plt.subplots(1, 2, figsize=(8, 4))

ax[0].imshow(cv2.cvtColor(test_img, cv2.COLOR_BGR2RGB))
ax[0].set_title("Original")
ax[0].axis("off")
if aligned is not None:
  ax[1].imshow(aligned)
  ax[1].set_title("Aligned")
  ax[1].axis("off")

else:
  ax[1].set_title("No face detected")
  ax[1].axis("off")

plt.show()

Even though RetinaFace cannot be reliably applied to this dataset without discarding valuable samples, it is still important to include a basic image enhancement step before running the BSIF feature extraction. Since BSIF encodes textural patterns through histogram representations, contrast enhancing techniques can be applied to ameliorate feature extraction. A particularly effective method would be the implementation of CLAHE enhancement.

In [ ]:
def clahe_enhancement(cv2_img):
  # this function uses CLAHE to enhance the input cv2 image
  clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
  img_clahe = clahe.apply(cv2_img)
  return img_clahe

### Features Extraction
Now that the image enhancing step has been defined the next stage is the implementation of BSIF for feature extraction. The BSIF method requires the use of a learned filter, for this task i have chosen a 7x7 12 bit map filter since its a good compromise.

In [ ]:
# ADDITIONAL PACKAGES INSTALLATION AND FILTERS DOWNLOAD

!pip install scipy

!gdown --id "1CPk07Oqn5gmKn8yXaEns-0yZc5ZAs3Su" -O texture_filters.zip
!unzip texture_filters.zip

clear_output()
print("Texture filters downloaded!")


Lets proceed with the BSIF definition and Feature extraction

In [ ]:
from scipy import signal
import scipy
import math

# ---------------- BSIF algorithm ----------------

def bsif(img, filterpath):
    f = scipy.io.loadmat(filterpath)
    texturefilters = f.get('ICAtextureFilters')

    # Initialize
    img = img.astype("float");
    numScl = np.shape(texturefilters)[2]
    codeImg = np.ones(np.shape(img))

    # Make spatial coordinates for sliding window
    r = int(math.floor(np.shape(texturefilters)[0] / 2))

    # Wrap image (increase image size according to maximum filter radius by wrapping around)
    upimg = img[0:r, :]
    btimg = img[-r:, :]
    lfimg = img[:, 0:r]
    rtimg = img[:, -r:]
    cr11 = img[0:r, 0:r]
    cr12 = img[0:r, -r:]
    cr21 = img[-r:, 0:r]
    cr22 = img[-r:, -r:]
    imgWrap = np.vstack(
        (np.hstack((cr22, btimg, cr21)), np.hstack((rtimg, img, lfimg)), np.hstack((cr12, upimg, cr11))))

    # Loop over scales
    for i in range(numScl):
        tmp = texturefilters[:, :, numScl - i - 1]
        ci = signal.convolve2d(imgWrap, np.rot90(tmp, 2), mode='valid')
        t = np.multiply(np.double(ci > 0), 2 ** i)
        codeImg = codeImg + t

    hist_bsif = np.histogram(codeImg.ravel(), bins=np.arange(1,(2**numScl)+2))
    hist_bsif = hist_bsif[0]
    # normalize the histogram
    hist_bsif = hist_bsif/(hist_bsif.sum() + 1e-7)


    return codeImg, hist_bsif

# ---------------- Data extraction functions ----------------

def bsif_data_extractor(dataset_path, filter_path):
  bsif_dataset_data = {}
  # load all images
  for fname in os.listdir(dataset_path):
      image = cv2.imread(os.path.join(dataset_path, fname))
      # Convert image to grayscale
      gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

      #and enhance with clahe
      clahe_enhanced = clahe_enhancement(gray_image)

      # extract bsif features
      image_bsif, hist = bsif(clahe_enhanced, filter_path)

      # create a dictionary containing image id and hist
      fname = fname.replace("_masked","")
      fname = fname.replace(".jpg","")
      bsif_dataset_data[fname] = hist
  return bsif_dataset_data

def bsif_probe_data_extractor(probe_path, filter_path):
  # extracs bsif computed representation for the probe image
  image = cv2.imread(probe_path)
  gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
  clahe_enhanced = clahe_enhancement(gray_image)
  image_bsif, hist = bsif(clahe_enhanced, filter_path)

  bsif_probe_data = hist

  return bsif_probe_data


Now that the algorithm has been defined, we can now start to extract data wich will be stored in feature vectors comprised by histograms.

In [ ]:
# ---------------- Files Location ----------------

filter_path = "../content/texturefilters/ICAtextureFilters_7x7_12bit.mat"
main_dataset_path = "../content/Masked_Recognition/Reference/"
probe_path = "../content/Masked_Recognition/probe.jpg"



# ---------------- Data extraction ----------------
bsif_dataset_data = bsif_data_extractor(main_dataset_path, filter_path)
bsif_probe_data = bsif_probe_data_extractor(probe_path, filter_path)

### Distances and Scores Computation
Once the BSIF feature vectors have been extracted, the next step is to compute distances and similarity scores in order to perform face recognition. The goal is to measure how close the probe image is to each reference sample in the gallery and then rank the identities accordingly.
To achieve this, a set of utility functions was first implemented to handle distance computation and score normalization.

The choosen metrics where:
- cosine similarity
- L2 normalized euclidean distance
- L1 normalized manhattan distance

In [ ]:
# ---------------- Models Output Wrapper ----------------

class OutputWrapper:
  # this is a simple class that i used to better store the scores
  # of the diffrent models implementation
    def __init__(self, distances, scores, best_id, best_distance):
        self.distances = distances
        self.scores = scores
        self.best_id = best_id
        self.best_distance = best_distance


# ---------------- Metrics ----------------

def l1_manhattan(x, y):
  # returns the l1 normalized manhattan distance between x and y
  return float(np.sum(np.abs(x - y)))

def l2_euclidean(x, y):
  # returns the l2 normalized euclidean distance between x and y
  return float(np.sqrt(np.sum(np.square(x - y))))

def cosine_similarity(x, y):
  # cosine similarity returns the l2 normalized cosine similarity
  # between x and y
  num = np.dot(x, y)
  denom = np.linalg.norm(x) * np.linalg.norm(y)
  if denom == 0: return 0
  else: return float(num / denom)

def cosine_distance(x, y):
  # this function is used to convert cosine similarity to a sort of
  # distance metric
  return float(1.0 - cosine_similarity(x, y))

# ---------------- Utilities ----------------

def values_normalization(scores: dict):
  # a simple normalization function that normalizes the score/distances values
  # contained inside the score/distances dicitonary
  # (wich should be structured as: {id : {scores/distances : accept/reject}})
  vals = [next(iter(inner.keys())) for inner in scores.values()]
  vmin, vmax = min(vals), max(vals)


  normalized = {}
  for id, inner in scores.items():
      dist = next(iter(inner.keys()))
      verify = next(iter(inner.values()))
      if vmax == vmin:
        # this comparison is used to give 0 as an output otherwise
        # the function will divide by 0 and the whole code will break
        norm_dist = 0.0
      else:
        norm_dist = (dist - vmin) / (vmax - vmin)

      normalized[id] = {norm_dist: verify}
  return normalized

def distances_to_score(scores: dict):
  # this function is the same as the one above, but the normalization is
  # inverted. Since this function is used to convert distances (the smaller the
  # better) to scores (the higher the better) the normalization of the distances
  # includes a step:
  #
  # score = 1.0 - (dist - vmin) / (vmax - vmin)
  #
  # to invert the ouput.

  vals = [next(iter(inner.keys())) for inner in scores.values()]
  vmax = max(vals)
  # vmin is set at 0.0 since this is the lowest distance possible between probe and sample
  vmin = 0.0

  score_dict = {}
  for id, inner in scores.items():
      dist = next(iter(inner.keys()))
      verify = next(iter(inner.values()))
      if vmax == vmin:
        # this comparison is used to give 0 as an output otherwise
        # the function will divide by 0 and the whole code will break
        score = 0.0
      else:
        score = 1.0 - (dist - vmin) / (vmax - vmin)

      score_dict[id] = {score: verify}
  return score_dict

def check_threshold(distance, threshold):
  # used to compute if the distance is less than the treshold
  if distance <= threshold:
    return True
  else:
    return False


def check_verify(verify_output : bool):
  # used in the score representation function to see if, based on the
  # threshold and verification ouput, the sample has been accepted
  # or rejected by the model
  if verify_output:
    return "Accept"
  else:
    return "Reject"

# ---------------- Results representation functions ----------------


def rank_by_score(scores : dict, score_type ="COSINE"):
  # this functions sor the scores by distances in a descending way
  # properly ranking by score the whole dataset vs probe results
  sorted_scores = sorted(
      scores.items(),
      key=lambda item: list(item[1].keys())[0]
  )

  print(f"\n----- {score_type} SCORES RANKING -----")
  for i in range(len(sorted_scores)):
      id = sorted_scores[-1 - i][0]
      score = list(sorted_scores[-1 - i][1].keys())[0]
      verify = sorted_scores[-1 - i][1][score]
      print(
          f"{i+1}) - [ID:{id}] "
          f"Computed prediction score: {round(score, 4)} -> {check_verify(verify)}"
      )
  print("\n")


def plt_best_correspondence(best_id, best_distance, type = "COSINE"):
  # ths function takes as an input the best results obtained by running the model
  # and compares them with the probe

  print("\n")
  fig, ax = plt.subplots(1, 2)

  best_id = best_id.replace("_masked","")
  best_correspondence = f"../content/Masked_Recognition/Reference/{best_id}.jpg"
  best_correspondence = cv2.imread(best_correspondence)
  best_correspondence = cv2.cvtColor(best_correspondence, cv2.COLOR_BGR2RGB)
  probe_img = cv2.imread("../content/Masked_Recognition/probe.jpg")
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)

  fig.suptitle(f"{type} LOWEST DISTANCE = {round(best_distance,4)}", fontweight="bold")

  ax[0].imshow(probe_img)
  ax[0].set_title("Probe Image")
  ax[0].axis("off")


  ax[1].imshow(best_correspondence)
  ax[1].set_title("Corresponding Identity")
  ax[1].axis("off")


  plt.show()


def plot_top10_scores(distances, scores , score_type="COSINE"):
    # this function sorts the distances and plots the top 10 scores/ distances
    # obtained.
    sorted_distances = dict(sorted(distances.items(), key=lambda x: next(iter(x[1]))))

    # Takes only the first 10 IDs
    top_ids = list(sorted_distances.keys())[:10]

    print("\n")
    fig2, axes = plt.subplots(2, 5, figsize=(20, 10))
    fig2.suptitle(f"\nTOP {len(top_ids)} {score_type} COMPUTED SCORES", fontweight="bold", fontsize=20)

    for i in range(10):
        ax = axes[i // 5, i % 5]
        ax.axis("off")

        image_id = str(top_ids[i]).replace("_masked", "")
        score = next(iter(sorted_distances[top_ids[i]]))

        img = cv2.imread(f"../content/Masked_Recognition/Reference/{image_id}.jpg")
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)

        ax.set_title(
            f"{i+1}- [ID:{image_id}]\n Distance: {round(score, 4)}\n Score: {round(next(iter(scores[image_id].keys())), 4)}",
            fontweight="bold"
        )

    plt.tight_layout()
    plt.show()

# ---------------- Performance evaluation functions ----------------

def labels_computation(true_identity_ids : list):
  # labels_computation takes as a input a list containing the true identites
  # inside the dataset and creates a "labels" dictioanry containing
  # {id : label }
  # the label will be either 1- for trye or 0- for false.
  # these label are used for performance computation
  labels = {}
  for fname in os.listdir("../content/Masked_Recognition/Reference/"):
    fname = fname.replace("_masked","")
    fname = fname.replace(".jpg","")
    if fname in true_identity_ids:
        labels.update({fname : 1})
    else:
        labels.update({fname : 0})
  return labels


def compute_far_frr(labels, distances):
  # compute_far_frr will compute treshold over distances for consisrency.
  # The treshold value in Deepface.verify is referring to the distance,
  # not the scores, so for consistency I am keeping this idea to see how
  # FAR and FRR vary over threshold distance increase.

  # fistly I compute the total number of possible impostor and genuine samples
  total_possible_impostors = len(labels) - sum(labels.values())
  total_possible_genuines = sum(labels.values())

  # here we normalize the distances in order to see how FAR and FRR varies by increasing the distance threshold
  distances = values_normalization(distances)
  FAR = []
  FRR = []

 # herer i included -0.01 and one value just above 1.0 to better see the FAR and FRR plotting
  thresholds = np.arange(-0.01, 1.01, 0.01)

  for threshold in thresholds:
    impostor = 0
    rejected_genuine = 0
    for id, values in distances.items():
      id = id.replace("_masked","")

      dist = next(iter(values.keys()))

      decision = check_threshold(dist,threshold)

      if decision and labels[id] == 0:
          impostor += 1
      elif not decision and labels[id] == 1:
          rejected_genuine += 1


    FAR.append(float(impostor/total_possible_impostors))
    FRR.append(float(rejected_genuine/total_possible_genuines))


  return thresholds, FAR, FRR


def plot_performances(labels: dict, distances: dict, score_type ="COSINE"):
    # plot_performances will use the computed FAR and FRR over the normalized
    # threshold (which spans form 0 to 1) to see their variation. This information
    # can be used to better tune the threshold value and see the implementation
    # performances.
    # EER and ROC are also represented but have no real infromational value for
    # out working case, since we re working on a single test of a probe image
    # vs a dataset containing only one sample per identity
    thresholds, FAR, FRR = compute_far_frr(labels, distances)

    FAR = np.asarray(FAR, dtype=float)
    FRR = np.asarray(FRR, dtype=float)

    # Percentages for plotting
    FAR_pct = FAR * 100.0
    FRR_pct = FRR * 100.0
    GAR_pct = (1.0 - FRR) * 100.0

    # EER
    diffs = np.abs(FAR - FRR)
    idx = int(np.argmin(diffs))
    EER = (FAR[idx] + FRR[idx]) / 2.0
    EER_th = thresholds[idx]
    print(f"\nThe Equal Error Rate (EER) computed with {score_type} distances is = {EER*100:.2f}%\n")

    # Plots
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"{score_type} METRICS PERFORMANCES", fontweight="bold", fontsize = 20)

    # FAR/FRR vs Threshold
    ax[0].plot(thresholds, FAR_pct, label='FAR (%)')
    ax[0].plot(thresholds, FRR_pct, label='FRR (%)')
    ax[0].set(xlabel='Threshold over distance', ylabel='Percentage (%)',
              title='FAR and FRR vs Threshold')
    ax[0].legend(); ax[0].grid(True)

    # ROC: GAR vs FAR
    ax[1].plot(FAR_pct, GAR_pct)
    ax[1].set(xlabel='FAR (%)', ylabel='GAR (%)', title='ROC (GAR vs FAR)')
    ax[1].grid(True)

    plt.show()



And now the proper score computation.

In [ ]:
# ---------------- Score computation functions ----------------
def compute_scores(bsif_dataset_data, bsif_probe_data, threshold=0.5):
    # This function compare dataset data bsif representation with probe data
    # bsif representation and computes the distances and scores using various
    # metrics.

    # here distances dictionaries are populated with id and sample-probe distance
    cosine_distances    = {id: cosine_distance(bsif_probe_data, bsif_sample) for id, bsif_sample in bsif_dataset_data.items()}
    euclidean_distances = {id: l2_euclidean(bsif_probe_data, bsif_sample) for id, bsif_sample in bsif_dataset_data.items()}
    manhattan_distances = {id: l1_manhattan(bsif_probe_data, bsif_sample) for id, bsif_sample in bsif_dataset_data.items()}


    # best matches are computed as the ones with the minimum probe-sample distance
    best_cosine_id,    best_cosine_distance    = min(cosine_distances.items(),    key=lambda kv: kv[1])
    best_euclidean_id, best_euclidean_distance = min(euclidean_distances.items(), key=lambda kv: kv[1])
    best_manhattan_id, best_manhattan_distance = min(manhattan_distances.items(), key=lambda kv: kv[1])

    print(f"Cosine computed distances: {cosine_distances}")
    print(f"Euclidean computed distances: {euclidean_distances}")
    print(f"Manhattan computed distances: {manhattan_distances}")



    # this cicle converts computed id-distance dictionaries
    # to usable {id : {distance : label}} dictionaries for compatibility with
    # all the previously defined utility functions.
    for id, distance in cosine_distances.items():
      cosine_distances[id] = {distance : check_threshold(distance, threshold)}
      euclidean_distances[id] = {euclidean_distances[id] : check_threshold(euclidean_distances[id],threshold)}
      manhattan_distances[id] = {manhattan_distances[id] : check_threshold(manhattan_distances[id],threshold)}

    best_cosine_verify    = cosine_distances[best_cosine_id][best_cosine_distance]
    best_euclidean_verify = euclidean_distances[best_euclidean_id][best_euclidean_distance]
    best_manhattan_verify = manhattan_distances[best_manhattan_id][best_manhattan_distance]
    print("\n")
    print(f"BEST COSINE CORRESPONENCE: ID = [{best_cosine_id}] DISTANCE: {best_cosine_distance} -> {check_verify(best_cosine_verify)}")
    print(f"BEST EUCLIDEAN CORRESPONENCE: ID = [{best_euclidean_id}] DISTANCE: {best_euclidean_distance} -> {check_verify(best_euclidean_verify)}")
    print(f"BEST MANHATTAN CORRESPONENCE: ID = [{best_manhattan_id}] DISTANCE: {best_manhattan_distance} -> {check_verify(best_manhattan_verify)}")


    # now we convert from distances to scores
    cosine_scores = distances_to_score(cosine_distances)
    euclidean_scores = distances_to_score(euclidean_distances)
    manhattan_scores = distances_to_score(manhattan_distances)


    return (cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance,\
            euclidean_distances,euclidean_scores, best_euclidean_id, best_euclidean_distance,\
            manhattan_distances,manhattan_scores, best_manhattan_id, best_manhattan_distance)

In [ ]:
# ---------------- Scores computation ----------------

cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance,\
euclidean_distances,euclidean_scores, best_euclidean_id, best_euclidean_distance,\
manhattan_distances,manhattan_scores, best_manhattan_id, best_manhattan_distance = compute_scores(bsif_dataset_data, bsif_probe_data)

bsif_cosine = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)
bsif_euclidean = OutputWrapper(euclidean_distances, euclidean_scores, best_euclidean_id, best_euclidean_distance)
bsif_manhattan = OutputWrapper(manhattan_distances, manhattan_scores, best_manhattan_id, best_manhattan_distance)

Lets represent the achieved results

In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(bsif_cosine.scores)
rank_by_score(bsif_euclidean.scores, "EUCLIDEAN")
rank_by_score(bsif_manhattan.scores, "MANHATTAN")

plt_best_correspondence(bsif_cosine.best_id, bsif_cosine.best_distance, "COSINE",)
plt_best_correspondence(bsif_euclidean.best_id, bsif_euclidean.best_distance, "EUCLIDEAN",)
plt_best_correspondence(bsif_manhattan.best_id, bsif_manhattan.best_distance,"MANHATTAN")

In [ ]:
# ---------------- TOP 10 scores per metric ----------------

plot_probe(probe_path)

plot_top10_scores(bsif_cosine.distances, bsif_cosine.scores, "COSINE")
plot_top10_scores(bsif_euclidean.distances, bsif_euclidean.scores, "EUCLIDEAN")
plot_top10_scores(bsif_manhattan.distances, bsif_manhattan.scores, "MANHATTAN")

We can already observe that the BSIF-based outputs are highly unreliable. Although the Manhattan distance provided slightly better results than the other tested metrics, the differences remain marginal. In practice, the computed distances between the probe and the dataset identities are extremely close to one another. As a result, even when the correct match is ranked first, the second-best candidate and the rest of the top-10 list have nearly identical distance values. This lack of separation produces inflated similarity scores across the board, leading to high uncertainty in the ranking.

To better illustrate the system's behavior, I computed the False Acceptance Rate and False Rejection Rate as functions of the decision threshold. This analysis allows for a more detailed examination of how the threshold value influences verification performance. I also generated the corresponding Equal Error Rate and Receiver Operating Characteristic curves. However these metrics are essentially uninformative in the current setup since the dataset contains only one probe image and a single gallery image per identity.

In [ ]:
# ---------------- Performances representation ----------------

labels = labels_computation(["100"])
plot_performances(labels, bsif_cosine.distances)
plot_performances(labels, bsif_euclidean.distances,"EUCLIDEAN")
plot_performances(labels, bsif_manhattan.distances,"MANHATTAN")

As previously said, and further confirmed by examining the FAR/FRR versus threshold plot, the Manhattan distance metric yields the most reliable performance within the BSIF implementation. Specifically, the distance computed between the probe and its true identity is consistently smaller than the distances to all impostor identities.

### Masked Implementation

Since the baseline BSIF algorithm proved to be rather unreliable, I explored whether focusing only on the non-occluded facial regions could improve recognition performance. The reasoning behid this is, because the probe image is masked, the lower half of the face provides no useful information for matching and may even introduce noise. By restricting feature extraction to the upper facial area, the algorithm can emphasize the regions that remain visible under mask occlusion.

To test this hypothesis, I created a synthetically masked version of the gallery dataset. Specifically, I implemented a preprocessing function that cuts the lower half of each identity image, effectively obscuring the region that would normally be hidden by a real mask. This ensures that both the probe and gallery images contain comparable visible areas.

Initially, the plan was to apply this masking step after alignment using RetinaFace, so that the synthetic masks would be placed consistently across all faces. However, since RetinaFace failed to detect certain samples in the dataset, this approach would have resulted in the loss of valuable gallery data. Instead, I applied the masking procedure directly on the original dataset images, taking advantage of the fact that they are already well aligned and have uniform dimensions. This guarantees consistency in how the masks are positioned across all samples, while retaining the full set of 81 identities.

In [ ]:

# ---------------- Utilities ----------------

def masked_area_remover(img_face, mask_area_percentage=0.50):
    # This function cuts out the lower part of the face image
    # and keeps only the visible region not occluded by the mask
    # mask_area_percentage sets the cut line, i have setted it at 0.5
    # which is just above the mask line.
    h, w = img_face.shape[:2]
    y_cut = int(h * mask_area_percentage)

    # Keep only the top region (not obscured)
    cropped = img_face[:y_cut, 0:w]

    return cropped

def masked_set_generation(new_folder_path, dataset_path, probe_path):
  os.makedirs(new_folder_path, exist_ok=True)

  process_counter = 0

  for fname in os.listdir(dataset_path):
    process_counter += 1
    print(f"processing image {fname} n #{process_counter}")
    img = cv2.imread(f"../content/Masked_Recognition/Reference/{fname}")
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    # here i used to apply the filter to the normalized images but since some faces
    # were not detected by the raw retinaface model i decided to apply the filter directly
    # to the starting dataset images since they are already aligned.

    masked_img = masked_area_remover(img)

    masked_out_path  = os.path.join(new_folder_path, f"{os.path.splitext(fname)[0]}_masked.jpg")
    cv2.imwrite(masked_out_path, cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB))

  clear_output()
  print(f"{process_counter} images have been processed!")

  # lets also mask the probe to check if accuracy will increase
  probe_img = cv2.imread(probe_path)
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)

  masked_img = masked_area_remover(probe_img)
  masked_out_path  = os.path.join("../content/Masked_Recognition/", "probe_masked.jpg")
  cv2.imwrite(masked_out_path, cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB))

In [ ]:
# ---------------- Folders ----------------
filter_path = "../content/texturefilters/ICAtextureFilters_7x7_12bit.mat"
masked_dataset_path ="../content/Masked_Recognition/Masked_References"
original_dataset_path =  "../content/Masked_Recognition/Reference"
probe_path = "../content/Masked_Recognition/probe.jpg"
masked_probe_path ="../content/Masked_Recognition/probe_masked.jpg"


#---------------- Masked set generation ----------------
masked_set_generation(masked_dataset_path, original_dataset_path, probe_path)

Now that the masked set has been generated I applied the BSIF algorithm.

In [ ]:

#---------------- Masked data extraction ----------------
bsif_dataset_data = bsif_data_extractor(masked_dataset_path, filter_path)
bsif_probe_data = bsif_probe_data_extractor(masked_probe_path, filter_path)


#---------------- Distances and score computation ----------------
cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance,\
euclidean_distances,euclidean_scores, best_euclidean_id, best_euclidean_distance,\
manhattan_distances,manhattan_scores, best_manhattan_id, best_manhattan_distance = compute_scores(bsif_dataset_data, bsif_probe_data)

bsif_cosine_masked = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)
bsif_euclidean_masked = OutputWrapper(euclidean_distances, euclidean_scores, best_euclidean_id, best_euclidean_distance)
bsif_manhattan_masked = OutputWrapper(manhattan_distances,manhattan_scores,best_manhattan_id, best_manhattan_distance)

In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(bsif_cosine_masked.scores)
rank_by_score(bsif_euclidean_masked.scores, "EUCLIDEAN")
rank_by_score(bsif_manhattan_masked.scores, "MANHATTAN")

plt_best_correspondence(bsif_cosine_masked.best_id, bsif_cosine_masked.best_distance, "COSINE",)
plt_best_correspondence(bsif_euclidean_masked.best_id, bsif_euclidean_masked.best_distance, "EUCLIDEAN",)
plt_best_correspondence(bsif_manhattan_masked.best_id, bsif_manhattan_masked.best_distance,"MANHATTAN")

In [ ]:
# ---------------- TOP 10 scores per metric ----------------
plot_probe(probe_path)

plot_top10_scores(bsif_cosine_masked.distances, bsif_cosine.scores, "COSINE")
plot_top10_scores(bsif_euclidean_masked.distances, bsif_euclidean.scores, "EUCLIDEAN")
plot_top10_scores(bsif_manhattan_masked.distances, bsif_manhattan.scores, "MANHATTAN")

In [ ]:
# ---------------- Performances representation ----------------

labels = labels_computation(["100"])
plot_performances(labels, bsif_cosine_masked.distances)
plot_performances(labels, bsif_euclidean_masked.distances,"EUCLIDEAN")
plot_performances(labels, bsif_manhattan_masked.distances,"MANHATTAN")

As observed from the experimental results, the introduction of artificial masking did not lead to a measurable improvement in performance. While the computed distances between the probe and the gallery samples became slightly closer than in the unmasked scenario, the relative separation between the true identity and the top impostor candidates remained minimal.

The lack of strong separation between genuine and impostor scores suggests that texture-based methods such as BSIF struggle to capture sufficiently robust information when only the visible half of the face is available.

## Deepface Implementation
Since the BSIF approach resulted in a highly unreliable recognition system under mask occlusion, the next logical step was to adopt a Convolutional Neural Network based recognizer.

However, the available dataset contains 81 identities with only a single reference image per class. This single sample per class scenario makes it impossible to train a custom CNN from scratch using frameworks such as Keras or PyTorch, as deep models require large amounts of data to generalize effectively.

To overcome this limitation, instead of training a new network, I employed [DeepFace](https://github.com/serengil/deepface?tab=readme-ov-file), a Python framework that integrates several pre-trained face recognition models and enhancers. Thanks to Deepface's modularity I was able to test different models, comparing their effectiveness when the probe image is partially occluded.

Since a wide range of tests were conducted in this scenario, it is not practical to present every single experiment in detail. Moreover, the computational cost of running DeepFace on the dataset is significant: on average, a full evaluation over the 81 gallery images with the selected probe requires approximately 20 minutes of processing time. To keep the discussion focused and avoid redundancy, I chose to present only the most relevant experiments and results



### ArcFace with masked dataset (basic)
In this implementation I selected the Facenet512 model, since it is reported in the DeepFace benchmark results (listed in the deepface github repository) as the best perfroming one.

For preprocessing, I enabled retinaface alignment, which in this case works reliably. Unlike the earlier standalone RetinaFace tests that failed on some dataset images, the embedded DeepFace retinaface implementation applies additional image enhancement steps before detection. Allowing retinaface to successfully align all the faces in the dataset without errors.

Watch out because this process is quite slow and will take approximatively 20 minutes to perform the image processing!

In [ ]:
!pip install deepface
clear_output()
print("Deepface has been installed!")

In [ ]:
import os
import numpy as np
from deepface import DeepFace

# ---------------- Main function ----------------
def deepface_model(gallery_path, probe_path, model, detector, threshold):
  # this function, based on deepface, takes as an input the whole reference
  # gallery, the probe path and the desired model detector and treshold in
  # order to leverage the Deepface.verify function and compare the masked probe
  # with all the identites inside the reference gallery.

  # Watch out, this function is quite slow and will take aproximately 20 min
  # on colab to process!

  process_counter = 0
  cosine_distances = {}

  for file in os.listdir(gallery_path):
    person_id = os.path.splitext(file)[0]

    # this .replace() passage is used by the masked_dataset implementation,
    # so it is present just for reusability sake.
    # It will do absolutely nothing when applied to the original reference set.
    person_id = person_id.replace("_masked","")
    img_path = os.path.join(gallery_path, file)

    # Deepface results are all tagged only with _cosine because all of
    # deepface top performing models (from arcface to facenet) are optimized
    # for distance computation with cosine metrics.
    result_cosine = DeepFace.verify(img_path, probe_path, model_name=model, detector_backend=detector, distance_metric = "cosine", threshold=0.5)
    cosine_distances.update({person_id: {result_cosine["distance"]: result_cosine["verified"]}})

    process_counter += 1
    print(f"processing image n #{process_counter}")


  clear_output()
  print(f"\nLoaded {len(cosine_distances)} identities into identity checker.")

  # Now we convert distances to proper score by normalizing the distances and flipping them
  cosine_scores = distances_to_score(cosine_distances)


  best_result = min(cosine_distances.items(), key=lambda values: list(values[1].keys())[0])
  best_cosine_id = best_result[0]
  best_cosine_distance = list(best_result[1].keys())[0]
  best_cosine_verify = check_verify(list(best_result[1].values())[0])


  print("-----DEEPFACE OUTPUT-----\n")
  print(f"scores : {cosine_scores}\ndistances: {cosine_distances}")
  print(f"\nBEST CORRESPONENCE: ID = [{best_cosine_id}] DISTANCE: {best_cosine_distance} -> {best_cosine_verify}")

  return (cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)



In [ ]:
# ---------------- Folders ----------------
masked_dataset_path ="../content/Masked_Recognition/Masked_References"
original_dataset_path =  "../content/Masked_Recognition/Reference"
probe_path = "../content/Masked_Recognition/probe.jpg"

#---------------- Masked Set Generation ----------------
masked_set_generation(masked_dataset_path, original_dataset_path, probe_path)


In [ ]:
# ---------------- Model Parameters ----------------
gallery_path = "../content/Masked_Recognition/Masked_References"
masked_probe_path   = "../content/Masked_Recognition/probe_masked.jpg"

MODEL_NAME = "Facenet512"
DETECTOR   = "retinaface"
THRESHOLD = 0.5

# ---------------- Run ----------------
cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance = deepface_model(
    gallery_path, masked_probe_path, MODEL_NAME, DETECTOR, THRESHOLD
)

deepface_face512_masked = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)

In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(deepface_face512_masked.scores)
plt_best_correspondence(deepface_face512_masked.best_id, deepface_face512_masked.best_distance, "COSINE",)


In [ ]:
# ---------------- TOP 10 scores per metric ----------------

plot_probe(probe_path)
plot_top10_scores(deepface_face512_masked.distances, deepface_face512_masked.scores, "COSINE")

In [ ]:
# ---------------- Performances representation ----------------

labels = labels_computation(["100"])
plot_performances(labels, deepface_face512_masked.distances)

The Facenet512 + RetinaFace implementation produced outstanding recognition performances. In contrast to the BSIF experiments, the distances between the probe and the gallery identities are now well separated, which significantly reduces uncertainty in the ranking. The genuine identity consistently appears as the top match, with a large margin in score difference compared to the nearest impostors, providing a high robustness for this implementation.

The only limitation of this approach lies in its computational efficiency, resulting in slow processing times, taking approximately 20 minutes to evaluate the dataset in its entirety. This slow processing time represents a real limitation of the current approach. In fact, the dataset used for this project is relatively small, consisting of only 81 gallery images plus the probe. Even with such a limited size, the complete evaluation requires close to 20 minutes of computation. In a practical deployment scenario with a significantly larger gallery the computation time would increase proportionally, making the current implementation impractical.

### ArcFace with masked dataset (enhanced)

During the previous experiments I observed that the distance computation itself is relatively fast, which suggested that the main bottleneck in the DeepFace execution was not the similarity calculation but rather the data extraction and additional overhead associated with the DeepFace.verify() function. This function performs multiple tasks internally, including preprocessing, representation, distance calculation, and thresholding, which increases execution time significantly.

In order to try to reduce processing time, I modified the workflow by using the DeepFace.represent() function instead. This function directly returns the feature representation leaving the computation of distances and similarity scores to be handled externally.

In [ ]:
# ---------------- Model Parameters ----------------
gallery_path = "../content/Masked_Recognition/Masked_References"
probe_path   = "../content/Masked_Recognition/probe_masked.jpg"

MODEL_NAME = "Facenet512"
DETECTOR   = "retinaface"
THRESHOLD = 0.5

# ---------------- Extractor Function ----------------

def deepface_extraction(gallery_path, probe_path, model, detector):
  # This function extracts the data representation list generated by deepface models
  # by leveraging Deepface.represent function and saves it inside
  # dataset_data and probe_data
  process_counter = 0
  dataset_data = {}
  for fname in os.listdir(gallery_path):
    feature_representation = DeepFace.represent(img_path = os.path.join(gallery_path, fname), model_name = model, detector_backend = detector)
    fname = fname.replace("_masked","")
    fname = fname.replace(".jpg","")
    dataset_data.update({fname : feature_representation})

    process_counter += 1
    print(f"processing image n #{process_counter}")

  clear_output()
  print(f"\nLoaded {len(dataset_data)} identities into identity checker.")

  print(f"\nprocessing probe...")
  probe_data = DeepFace.represent(img_path = probe_path, model_name = model, detector_backend = detector)


  clear_output()
  print(f"Dataset and Probe processed!")
  return dataset_data, probe_data

In [ ]:
# ---------------- Scores computation functions ----------------

def extract_data_representation(data):
  # since Deepface.represent generates a list which contains a dicitonray
  # structured as { "embedding" : [data representation], facial_area : ... }
  # I used this function to extract the image representation generated by deepface.
  lista = data[0]
  return lista["embedding"]



def compute_deepface_scores(dataset_data, probe_data, threshold=0.5):
    # compute_deepface_scores computes cosine similarity between samples and the
    # probe since Deepface representation as said before is optimized for
    # cosine similarity metrics.
    probe_vec = extract_data_representation(probe_data)

    # compute similarity distances
    cosine_distances = {
        id: cosine_distance(probe_vec, extract_data_representation(dataset_sample))
        for id, dataset_sample in dataset_data.items()
    }

    # convert computed id-distance pairs to usable {id : {distance : label}} dictionaries
    for id, distance in cosine_distances.items():
      cosine_distances[id] = {distance : check_threshold(distance, threshold)}


    # best match have the minimum distance
    print(cosine_distances)
    best_result = min(cosine_distances.items(), key=lambda values: list(values[1].keys())[0])

    # and now we compute the scores
    cosine_scores = distances_to_score(cosine_distances)


    print(f"Cosine distances: {cosine_distances}")
    print(f"Cosine scores: {cosine_scores}")

    best_result = min(cosine_distances.items(), key=lambda values: list(values[1].keys())[0])
    best_cosine_id = best_result[0]
    best_cosine_distance = list(best_result[1].keys())[0]
    best_cosine_verify = check_verify(list(best_result[1].values())[0])

    print("\n")
    print(f"BEST COSINE CORRESPONENCE: ID = [{best_cosine_id}] DISTANCE: {best_cosine_distance} -> {check_verify(best_cosine_verify)}")

    return (cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)



In [ ]:
# ---------------- Data extraction ----------------

dataset_data, probe_data = deepface_extraction(gallery_path, probe_path, MODEL_NAME, DETECTOR)

# ---------------- Scores computation ----------------

cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance = compute_deepface_scores(dataset_data, probe_data, 0.5)
deepface_face512_masked_speed = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)


In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(deepface_face512_masked.scores)
plt_best_correspondence(deepface_face512_masked_speed.best_id, deepface_face512_masked_speed.best_distance, "COSINE",)


In [ ]:
# ---------------- TOP 10 scores per metric ----------------

plot_probe(probe_path)
plot_top10_scores(deepface_face512_masked_speed.distances, deepface_face512_masked_speed.scores, "COSINE")

In [ ]:
# ---------------- Performances representation ----------------

labels = labels_computation(["100"])
plot_performances(labels, deepface_face512_masked_speed.distances)

Through this final implementation, we achieved a processing speed-up of approximately 10 minutes compared to the previous approach, halving the processing time, while maintaining identical recognition results.

### Non masked comparison
Lets show if the focusing in non occluded area is effectively improving the performances

In [ ]:

# ---------------- Model Parameters ----------------
gallery_path = "../content/Masked_Recognition/Reference"
probe_path   = "../content/Masked_Recognition/probe.jpg"

MODEL_NAME = "Facenet512"
DETECTOR   = "retinaface"
THRESHOLD = 0.5
# ---------------- Data extraction ----------------

dataset_data, probe_data = deepface_extraction(gallery_path, probe_path, MODEL_NAME, DETECTOR)

# ---------------- Scores computation ----------------

cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance = compute_deepface_scores(dataset_data, probe_data, 0.5)
deepface_face512_basic_speed = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)


In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(deepface_face512_basic_speed.scores)
plt_best_correspondence(deepface_face512_basic_speed.best_id, deepface_face512_basic_speed.best_distance, "COSINE",)


In [ ]:
# ---------------- TOP 10 scores per metric ----------------

plot_probe(probe_path)
plot_top10_scores(deepface_face512_basic_speed.distances, deepface_face512_basic_speed.scores, "COSINE")

In [ ]:
# ---------------- Performances representation ----------------

labels = labels_computation(["100"])
plot_performances(labels, deepface_face512_basic_speed.distances)

We can clearly observe that masking the lower part of the gallery images improves performance compared to using the unmasked dataset. In particular, the distances computed between the probe and its true identity are consistently lower in the artificially masked scenario than the ones reported above.

#Final Implementation

## Utilities
To ensure the correct execution of the entire face recognition pipeline, a number of utility functions and external packages are required.

In [ ]:
# PACKAGES INSTALLATION

from IPython.display import clear_output

!pip install pydrive
!pip install dlib
!pip install opencv-python
!pip install scikit-image
!pip install deepface

clear_output()

print("Packages have been installed!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import os
import dlib
import os
from deepface import DeepFace
import imutils

!gdown --id "11V4qLWcpa2Njumt1g8t44sBqe4IBnk0K" -O faceRecognDataset.zip
!unzip faceRecognDataset.zip


clear_output()
print("Files have been downloaded!")

In [ ]:
# ---------------- Models Output Wrapper ----------------

class OutputWrapper:
  # this is a simple class that i used to better store the scores
  # of the diffrent models implementation
    def __init__(self, distances, scores, best_id, best_distance):
        self.distances = distances
        self.scores = scores
        self.best_id = best_id
        self.best_distance = best_distance


# ---------------- Metrics ----------------

def l1_manhattan(x, y):
  # returns the l1 normalized manhattan distance between x and y
  return float(np.sum(np.abs(x - y)))

def l2_euclidean(x, y):
  # returns the l2 normalized euclidean distance between x and y
  return float(np.sqrt(np.sum(np.square(x - y))))

def cosine_similarity(x, y):
  # cosine similarity returns the l2 normalized cosine similarity
  # between x and y
  num = np.dot(x, y)
  denom = np.linalg.norm(x) * np.linalg.norm(y)
  if denom == 0: return 0
  else: return float(num / denom)

def cosine_distance(x, y):
  # this function is used to convert cosine similarity to a sort of
  # distance metric
  return float(1.0 - cosine_similarity(x, y))

# ---------------- Utilities ----------------

def values_normalization(scores: dict):
  # a simple normalization function that normalizes the score/distances values
  # contained inside the score/distances dicitonary
  # (wich should be structured as: {id : {scores/distances : accept/reject}})
  vals = [next(iter(inner.keys())) for inner in scores.values()]
  vmin, vmax = min(vals), max(vals)


  normalized = {}
  for id, inner in scores.items():
      dist = next(iter(inner.keys()))
      verify = next(iter(inner.values()))
      if vmax == vmin:
        # this comparison is used to give 0 as an output otherwise
        # the function will divide by 0 and the whole code will break
        norm_dist = 0.0
      else:
        norm_dist = (dist - vmin) / (vmax - vmin)

      normalized[id] = {norm_dist: verify}
  return normalized

def distances_to_score(scores: dict):
  # this function is the same as the one above, but the normalization is
  # inverted. Since this function is used to convert distances (the smaller the
  # better) to scores (the higher the better) the normalization of the distances
  # includes a step:
  #
  # score = 1.0 - (dist - vmin) / (vmax - vmin)
  #
  # to invert the ouput.

  vals = [next(iter(inner.keys())) for inner in scores.values()]
  vmax = max(vals)
  # vmin is set at 0.0 since this is the lowest distance possible between probe and sample
  vmin = 0.0

  score_dict = {}
  for id, inner in scores.items():
      dist = next(iter(inner.keys()))
      verify = next(iter(inner.values()))
      if vmax == vmin:
        # this comparison is used to give 0 as an output otherwise
        # the function will divide by 0 and the whole code will break
        score = 0.0
      else:
        score = 1.0 - (dist - vmin) / (vmax - vmin)

      score_dict[id] = {score: verify}
  return score_dict

def check_threshold(distance, threshold):
  # used to compute if the distance is less than the treshold
  if distance <= threshold:
    return True
  else:
    return False


def check_verify(verify_output : bool):
  # used in the score representation function to see if, based on the
  # threshold and verification ouput, the sample has been accepted
  # or rejected by the model
  if verify_output:
    return "Accept"
  else:
    return "Reject"

# ---------------- Results representation functions ----------------

def plot_probe(probe_path):
  # this utiltiy function reads the input image through cv2 and plots it with
  # matplotlib pyplot.

  probe_img = cv2.imread(probe_path)
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)
  fig1, axes = plt.subplots(1,figsize=(3, 3))
  axes.imshow(probe_img)
  axes.set_title("Probe Image", fontweight="bold")
  axes.axis("off")
  plt.show()
  print("\n")

def rank_by_score(scores : dict, score_type ="COSINE"):
  # this functions sor the scores by distances in a descending way
  # properly ranking by score the whole dataset vs probe results
  sorted_scores = sorted(
      scores.items(),
      key=lambda item: list(item[1].keys())[0]
  )

  print(f"\n----- {score_type} SCORES RANKING -----")
  for i in range(len(sorted_scores)):
      id = sorted_scores[-1 - i][0]
      score = list(sorted_scores[-1 - i][1].keys())[0]
      verify = sorted_scores[-1 - i][1][score]
      print(
          f"{i+1}) - [ID:{id}] "
          f"Computed prediction score: {round(score, 4)} -> {check_verify(verify)}"
      )
  print("\n")


def plt_best_correspondence(best_id, best_distance, type = "COSINE"):
  # ths function takes as an input the best results obtained by running the model
  # and compares them with the probe

  print("\n")
  fig, ax = plt.subplots(1, 2)

  best_id = best_id.replace("_masked","")
  best_correspondence = f"../content/Masked_Recognition/Reference/{best_id}.jpg"
  best_correspondence = cv2.imread(best_correspondence)
  best_correspondence = cv2.cvtColor(best_correspondence, cv2.COLOR_BGR2RGB)
  probe_img = cv2.imread("../content/Masked_Recognition/probe.jpg")
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)

  fig.suptitle(f"{type} LOWEST DISTANCE = {round(best_distance,4)}", fontweight="bold")

  ax[0].imshow(probe_img)
  ax[0].set_title("Probe Image")
  ax[0].axis("off")


  ax[1].imshow(best_correspondence)
  ax[1].set_title("Corresponding Identity")
  ax[1].axis("off")


  plt.show()


def plot_top10_scores(distances, scores , score_type="COSINE"):
    # this function sorts the distances and plots the top 10 scores/ distances
    # obtained.
    sorted_distances = dict(sorted(distances.items(), key=lambda x: next(iter(x[1]))))

    # Takes only the first 10 IDs
    top_ids = list(sorted_distances.keys())[:10]

    print("\n")
    fig2, axes = plt.subplots(2, 5, figsize=(20, 10))
    fig2.suptitle(f"\nTOP {len(top_ids)} {score_type} COMPUTED SCORES", fontweight="bold", fontsize=20)

    for i in range(10):
        ax = axes[i // 5, i % 5]
        ax.axis("off")

        image_id = str(top_ids[i]).replace("_masked", "")
        score = next(iter(sorted_distances[top_ids[i]]))

        img = cv2.imread(f"../content/Masked_Recognition/Reference/{image_id}.jpg")
        if img is not None:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            ax.imshow(img)

        ax.set_title(
            f"{i+1}- [ID:{image_id}]\n Distance: {round(score, 4)}\n Score: {round(next(iter(scores[image_id].keys())), 4)}",
            fontweight="bold"
        )

    plt.tight_layout()
    plt.show()

# ---------------- Performance evaluation functions ----------------

def labels_computation(true_identity_ids : list):
  # labels_computation takes as a input a list containing the true identites
  # inside the dataset and creates a "labels" dictioanry containing
  # {id : label }
  # the label will be either 1- for trye or 0- for false.
  # these label are used for performance computation
  labels = {}
  for fname in os.listdir("../content/Masked_Recognition/Reference/"):
    fname = fname.replace("_masked","")
    fname = fname.replace(".jpg","")
    if fname in true_identity_ids:
        labels.update({fname : 1})
    else:
        labels.update({fname : 0})
  return labels


def compute_far_frr(labels, distances):
  # compute_far_frr will compute treshold over distances for consisrency.
  # The treshold value in Deepface.verify is referring to the distance,
  # not the scores, so for consistency I am keeping this idea to see how
  # FAR and FRR vary over threshold distance increase.

  # fistly I compute the total number of possible impostor and genuine samples
  total_possible_impostors = len(labels) - sum(labels.values())
  total_possible_genuines = sum(labels.values())

  # here we normalize the distances in order to see how FAR and FRR varies by increasing the distance threshold
  distances = values_normalization(distances)
  FAR = []
  FRR = []

 # herer i included -0.01 and one value just above 1.0 to better see the FAR and FRR plotting
  thresholds = np.arange(-0.01, 1.01, 0.01)

  for threshold in thresholds:
    impostor = 0
    rejected_genuine = 0
    for id, values in distances.items():
      id = id.replace("_masked","")

      dist = next(iter(values.keys()))

      decision = check_threshold(dist,threshold)

      if decision and labels[id] == 0:
          impostor += 1
      elif not decision and labels[id] == 1:
          rejected_genuine += 1


    FAR.append(float(impostor/total_possible_impostors))
    FRR.append(float(rejected_genuine/total_possible_genuines))


  return thresholds, FAR, FRR


def plot_performances(labels: dict, distances: dict, score_type ="COSINE"):
    # plot_performances will use the computed FAR and FRR over the normalized
    # threshold (which spans form 0 to 1) to see their variation. This information
    # can be used to better tune the threshold value and see the implementation
    # performances.
    # EER and ROC are also represented but have no real infromational value for
    # out working case, since we re working on a single test of a probe image
    # vs a dataset containing only one sample per identity
    thresholds, FAR, FRR = compute_far_frr(labels, distances)

    FAR = np.asarray(FAR, dtype=float)
    FRR = np.asarray(FRR, dtype=float)

    # Percentages for plotting
    FAR_pct = FAR * 100.0
    FRR_pct = FRR * 100.0
    GAR_pct = (1.0 - FRR) * 100.0

    # EER
    diffs = np.abs(FAR - FRR)
    idx = int(np.argmin(diffs))
    EER = (FAR[idx] + FRR[idx]) / 2.0
    EER_th = thresholds[idx]
    print(f"\nThe Equal Error Rate (EER) computed with {score_type} distances is = {EER*100:.2f}%\n")

    # Plots
    fig, ax = plt.subplots(1, 2, figsize=(15, 6))
    fig.suptitle(f"{score_type} METRICS PERFORMANCES", fontweight="bold", fontsize = 20)

    # FAR/FRR vs Threshold
    ax[0].plot(thresholds, FAR_pct, label='FAR (%)')
    ax[0].plot(thresholds, FRR_pct, label='FRR (%)')
    ax[0].set(xlabel='Threshold over distance', ylabel='Percentage (%)',
              title='FAR and FRR vs Threshold')
    ax[0].legend(); ax[0].grid(True)

    # ROC: GAR vs FAR
    ax[1].plot(FAR_pct, GAR_pct)
    ax[1].set(xlabel='FAR (%)', ylabel='GAR (%)', title='ROC (GAR vs FAR)')
    ax[1].grid(True)

    plt.show()



# ---------------- Artificial mask generation ----------------

def masked_area_remover(img_face, mask_area_percentage=0.50):
    # This function cuts out the lower part of the face image
    # and keeps only the visible region not occluded by the mask
    # mask_area_percentage sets the cut line, i have setted it at 0.5
    # which is just above the mask line.
    h, w = img_face.shape[:2]
    y_cut = int(h * mask_area_percentage)

    # Keep only the top region (not obscured)
    cropped = img_face[:y_cut, 0:w]

    return cropped

def masked_set_generation(new_folder_path, dataset_path, probe_path):
  os.makedirs(new_folder_path, exist_ok=True)

  process_counter = 0

  for fname in os.listdir(dataset_path):
    process_counter += 1
    print(f"processing image {fname} n #{process_counter}")
    img = cv2.imread(f"../content/Masked_Recognition/Reference/{fname}")
    img = cv2.cvtColor(img,cv2.COLOR_BGR2RGB)

    # here i used to apply the filter to the normalized images but since some faces
    # were not detected by the raw retinaface model i decided to apply the filter directly
    # to the starting dataset images since they are already aligned.

    masked_img = masked_area_remover(img)

    masked_out_path  = os.path.join(new_folder_path, f"{os.path.splitext(fname)[0]}_masked.jpg")
    cv2.imwrite(masked_out_path, cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB))

  clear_output()
  print(f"{process_counter} images have been processed!")

  # lets also mask the probe to check if accuracy will increase
  probe_img = cv2.imread(probe_path)
  probe_img = cv2.cvtColor(probe_img, cv2.COLOR_BGR2RGB)

  masked_img = masked_area_remover(probe_img)
  masked_out_path  = os.path.join("../content/Masked_Recognition/", "probe_masked.jpg")
  cv2.imwrite(masked_out_path, cv2.cvtColor(masked_img, cv2.COLOR_BGR2RGB))


##1- Dataset Preparation and Analysis

In [ ]:
# lets plot the probe image
probe_path = "../content/Masked_Recognition/probe.jpg"
plot_probe(probe_path)

# and the total number of reference inside the dataset
num_images = len(os.listdir("../content/Masked_Recognition/Reference/"))
print(f"\nThe total numer of images inside the reference dataset is: {num_images}")

##2- Implementation of the recognition system
Now that we have determined the probe image and the fact that 81 identities with 1 sample each are present inside the dataset lets implement a recognition system.

As shown inside the Sperimentation section, we obtained the best results with a Deepface based implementation with retinaface enhancement and Facenet512 model, while focusing on the non occluded sections of the face.

In [ ]:
# ---------------- Folders ----------------
masked_dataset_path ="../content/Masked_Recognition/Masked_References"
original_dataset_path =  "../content/Masked_Recognition/Reference"
probe_path = "../content/Masked_Recognition/probe.jpg"

#---------------- Masked Set Generation ----------------
masked_set_generation(masked_dataset_path, original_dataset_path, probe_path)


Now lets implement the proper face recognition system

In [ ]:
# ---------------- Model Parameters ----------------
gallery_path = "../content/Masked_Recognition/Masked_References"
masked_probe_path   = "../content/Masked_Recognition/probe_masked.jpg"
MODEL_NAME = "Facenet512"
DETECTOR   = "retinaface"
THRESHOLD = 0.5

# ---------------- Extractor Function ----------------

def deepface_extraction(gallery_path, probe_path, model, detector):
  # This function extracts the data representation list generated by deepface models
  # by leveraging Deepface.represent function and saves it inside
  # dataset_data and probe_data
  process_counter = 0
  dataset_data = {}
  for fname in os.listdir(gallery_path):
    feature_representation = DeepFace.represent(img_path = os.path.join(gallery_path, fname), model_name = model, detector_backend = detector)
    fname = fname.replace("_masked","")
    fname = fname.replace(".jpg","")
    dataset_data.update({fname : feature_representation})

    process_counter += 1
    print(f"processing image n #{process_counter}")

  clear_output()
  print(f"\nLoaded {len(dataset_data)} identities into identity checker.")

  print(f"\nprocessing probe...")
  probe_data = DeepFace.represent(img_path = probe_path, model_name = model, detector_backend = detector)


  clear_output()
  print(f"Dataset and Probe processed!")
  return dataset_data, probe_data

# ---------------- Scores computation functions ----------------

def extract_data_representation(data):
  # since Deepface.represent generates a list which contains a dicitonray
  # structured as { "embedding" : [data representation], facial_area : ... }
  # I used this function to extract the image representation generated by deepface.
  lista = data[0]
  return lista["embedding"]



def compute_deepface_scores(dataset_data, probe_data, threshold=0.5):
    # compute_deepface_scores computes cosine similarity between samples and the
    # probe since Deepface representation as said before is optimized for
    # cosine similarity metrics.
    probe_vec = extract_data_representation(probe_data)

    # compute similarity distances
    cosine_distances = {
        id: cosine_distance(probe_vec, extract_data_representation(dataset_sample))
        for id, dataset_sample in dataset_data.items()
    }

    # convert computed id-distance pairs to usable {id : {distance : label}} dictionaries
    for id, distance in cosine_distances.items():
      cosine_distances[id] = {distance : check_threshold(distance, threshold)}


    # best match have the minimum distance
    print(cosine_distances)
    best_result = min(cosine_distances.items(), key=lambda values: list(values[1].keys())[0])

    # and now we compute the scores
    cosine_scores = distances_to_score(cosine_distances)


    print(f"Cosine distances: {cosine_distances}")
    print(f"Cosine scores: {cosine_scores}")

    best_result = min(cosine_distances.items(), key=lambda values: list(values[1].keys())[0])
    best_cosine_id = best_result[0]
    best_cosine_distance = list(best_result[1].keys())[0]
    best_cosine_verify = check_verify(list(best_result[1].values())[0])

    print("\n")
    print(f"BEST COSINE CORRESPONENCE: ID = [{best_cosine_id}] DISTANCE: {best_cosine_distance} -> {check_verify(best_cosine_verify)}")

    return (cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)


##2- Results analysis
Now that the whole system has been implemented we just need to analyze the results.

In [ ]:
# ---------------- Data extraction ----------------

dataset_data, probe_data = deepface_extraction(gallery_path, masked_probe_path, MODEL_NAME, DETECTOR)

# ---------------- Scores computation ----------------

cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance = compute_deepface_scores(dataset_data, probe_data, 0.5)
deepface_face512_enh_speed = OutputWrapper(cosine_distances, cosine_scores, best_cosine_id, best_cosine_distance)

In [ ]:
# ---------------- Rankings and Best Correspondence per metric ----------------

rank_by_score(deepface_face512_enh_speed.scores)
plt_best_correspondence(deepface_face512_enh_speed.best_id, deepface_face512_enh_speed.best_distance, "COSINE",)

In [ ]:
# ---------------- TOP 10 scores per metric ----------------

plot_probe(probe_path)
plot_top10_scores(deepface_face512_enh_speed.distances, deepface_face512_enh_speed.scores, "COSINE")

To answer the following questions:
- Can any possible correspondence be visually confirmed despite the occlusion?
- Does focusing on non-occluded regions improve recognition performance?

For the first question we can clearly say Yes. Even though the probe image is partially occluded by a mask, both visual inspection and the recognition system confirm that the probe identity corresponds to ID = 100. The distinctive features in the non-occluded regions provide enough information for both us and the algorithm to establish the match.

While for the second we can also say that it is the case, as demonstrated in the experimentation section. When the dataset was artificially masked so that only the upper face remained visible, the distances between the probe representation and its true identity decreased.